# Extract necessary columns from raw spss

In [2]:
cols_explotaciones = [
    "S101", "S102", "S106", "S108", 
    "S211D", "S324", "S1067", "S1068A", 
    "S1069A", "S1170A", "S1273", "S1274B1", 
    "S1274B2", "S1275A", "S1275C", "S1275G", 
    "S1579", "S1683", "S1685F"
]

cols_parcelas = [
    "S101", "S102", "S106", "S108",
    "S434A", "S434B", "S434C", "S434D", 
    "S434E", "S434F", "S434G", "S434H"
]

explotaciones_df = pd.read_spss("data/cenagro-2011-explotaciones-agropecuarias.sav", usecols = cols_explotaciones)
aprovechamiento_df = pd.read_spss("data/cenagro-2011-parcelas-aprovechamiento-tierra.sav", usecols = cols_parcelas)

In [3]:
import json
from datetime import datetime

def datetime_handler(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")

def save_to_parquet_file(df, name, outdir=""):
    with open(f"{outdir}/{name}-metadata.json", "w", encoding="utf-8") as f:
        json.dump(df.attrs, f, indent=4, default=datetime_handler)
    df.attrs = {}
    df.to_parquet(f"{outdir}/{name}.parquet", engine='pyarrow', index=False) 


In [ ]:


del explotaciones_df
del aprovechamiento_df

# Create ducksql database

In [5]:
import duckdb
import os

db_path = "data/cenagro.duckdb"
os.remove(db_path)


conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE IF NOT EXISTS fact_explotaciones AS 
    SELECT 
        CAST(S101 AS VARCHAR) AS id_departamento,
        CAST(S102 AS VARCHAR) AS id_municipio,
        CAST(S106 AS INTEGER) AS id_sea,
        CAST(S108 AS INTEGER) AS id_explotacion,
        
        S211D AS sexo_productor,
        S1170A AS asistencia_tecnica,
        S324 AS orientacion_mercado,
        S1579 AS tiene_ingreso_extra,
        S1067 AS contrato_trabajadores,
        S1068A AS trabajadores_permanentes,
        S1069A AS trabajadores_temporales,
        S1273 AS usa_credito,
        S1275A AS id_fuente_banco,
        S1275C AS id_fuente_ong,
        S1275G AS id_fuente_prestamista,
        S1683 AS escasez_alimentos,
        S1685F AS venta_activos
        
    FROM read_parquet('data/explotaciones.parquet')
""")

conn.execute("""
    CREATE TABLE IF NOT EXISTS dim_superficie_finca AS 
    SELECT 
        S101 AS id_departamento,
        S102 AS id_municipio,
        S106 AS id_sea,
        S108 AS id_explotacion,
        SUM(
            COALESCE(S434A, 0) + 
            COALESCE(S434B, 0) + 
            COALESCE(S434C, 0) + 
            COALESCE(S434D, 0) + 
            COALESCE(S434E, 0) + 
            COALESCE(S434F, 0) + 
            COALESCE(S434G, 0) + 
            COALESCE(S434H, 0)
        ) AS superficie_total_manzanas
    FROM read_parquet('data/parcelas.parquet')
    GROUP BY 
        S101, S102, S106, S108
""")


conn.execute("CREATE UNIQUE idx_fact_pk ON fact_explotaciones (id_departamento, id_municipio, id_sea, id_explotacion)")
conn.execute("CREATE UNIQUE idx_dim_superficie_pk ON dim_superficie_finca (id_departamento, id_municipio, id_sea, id_explotacion)")

conn.close()

ParserException: Parser Error: syntax error at or near "idx_fact_pk"

LINE 1: CREATE UNIQUE idx_fact_pk ON fact_explotaciones (id_departamento, id_muni...
                      ^